In [ ]:
# Not using right now, but this is how to stream the dataset and display it in a DataFrame
from datasets import load_dataset
import pandas as pd

# Stream the dataset
ds = load_dataset("OpenPipe/hacker-news", split="train", streaming=True)

# Number of posts you want
N = 10
sample_posts = []

for post in ds:
    # Only keep posts with non-empty text
    text = post.get("text")
    if text and text.strip():
        sample_posts.append(post)
    if len(sample_posts) >= N:
        break

# Convert to DataFrame
df = pd.DataFrame(sample_posts)

# Keep only columns that exist
#desired_cols = ["title", "url", "text", "time"]
#cols_to_keep = [c for c in desired_cols if c in df.columns]
#df = df[cols_to_keep]

# Optional: shorten text for display
df["text"] = df["text"].str.slice(0, 300)

# Convert timestamp to string if it's not already
if "time" in df.columns:
    df["time"] = df["time"].apply(
        lambda t: t.strftime('%Y-%m-%d %H:%M:%S') if hasattr(t, "strftime") else str(t)
    )

# Display nicely
df

Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

,id,type,by,time,title,text,url,score,parent,top_level_parent,descendants,kids,deleted,dead
0,15,comment,sama,2006-10-09 19:51:01,None,&#34;the rising star of venture capital&#34; -...,None,None,1,1,None,[17],None,None
1,17,comment,pg,2006-10-09 19:52:45,None,Is there anywhere to eat on Sandhill Road?,None,None,15,1,None,[1079],None,None
2,22,comment,pg,2006-10-10 02:18:22,None,It's kind of funny that Sevin Rosen is giving ...,None,None,21,21,None,None,None,None
3,23,comment,starklysnarky,2006-10-10 02:30:53,None,"This is interesting, but the limitations becom...",None,None,20,20,None,None,None,None
4,30,comment,spez,2006-10-10 15:34:59,None,Stay tuned...,None,None,29,29,None,[31],None,None
5,31,comment,pg,2006-10-10 15:40:05,None,I'm tuned...,None,None,30,29,None,[33],None,None
6,33,comment,spez,2006-10-10 15:50:40,None,winnar winnar chicken dinnar!,None,None,31,29,None,[34],None,None
7,34,comment,pg,2006-10-10 15:53:53,None,what do you mean? this story's still not #1,None,None,33,29,None,"[36, 35, 531706]",None,None
8,35,comment,spez,2006-10-10 15:57:42,None,perhaps if i hadn't told you it was coming\r\n...,None,None,34,29,None,None,None,None
9,36,comment,pg,2006-10-10 16:01:01,None,Can you do it again?,None,None,34,29,None,None,None,None


In [15]:
# Import media reliability dataset in full

media_ds = load_dataset("sergioburdisso/news_media_reliability", split="train")

media_df = pd.DataFrame(media_ds)

media_df

media_lookup = {
    row['domain']: {
        "reliability_label": row['reliability_label'],
        "newsguard_score": row['newsguard_score']
    }
    for _, row in media_df.iterrows()
}

In [ ]:
from datasets import load_dataset
import pandas as pd
import re
from dateutil import parser
from urllib.parse import urlparse
import random

# Stream the dataset
ds = load_dataset("ashraq/financial-news-articles", split="train", streaming=True)

# Company we want to look at
company = "all"   # change to "all" to pull everything

# Options
sample = True       # whether to sample the matched articles
sample_frac = 0.1   # fraction to keep if sampling
# N = 10            # optional: max number of collected articles

sample_articles = []

def extract_domain(url):
    if not url:
        return None
    return urlparse(url).netloc.replace("www.", "")

def extract_date(text):
    if not text:
        return None

    date_patterns = [
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4}',
        r'\b\d{4}-\d{2}-\d{2}',
        r'\b\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{4}'
    ]

    for pattern in date_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            try:
                return parser.parse(match.group(), fuzzy=True)
            except:
                continue

    return None

company_lower = company.lower()

for article in ds:
    headline = article.get("title", "")
    text = article.get("text", "")

    matches = (
        company_lower == "all"
        or company_lower in headline.lower()
        or company_lower in text.lower()
    )

    if matches:
        # Add news_site
        article["news_site"] = extract_domain(article.get("url"))

        # Add extracted dates
        article["title_date"] = extract_date(headline)
        article["text_date"] = extract_date(text)
        
        domain = article["news_site"]
        if domain in media_lookup:
            article["reliability_label"] = media_lookup[domain]["reliability_label"]
            article["newsguard_score"] = media_lookup[domain]["newsguard_score"]
        else:
            article["reliability_label"] = None
            article["newsguard_score"] = None

        sample_articles.append(article)

# Optional sampling after streaming
if sample and sample_articles:
    k = max(1, int(len(sample_articles) * sample_frac))
    sample_articles = random.sample(sample_articles, k)

# Convert to DataFrame (optional, just for viewing)
df = pd.DataFrame(sample_articles)

df

,title,text,url,news_site,title_date,text_date,reliability_label,newsguard_score
0,"Burundi bans the BBC, VOA two weeks before ref...","NAIROBI, May 4 (Reuters) - Burundi suspended o...",https://www.reuters.com/article/burundi-politi...,reuters.com,None,None,1.0,100.0
1,Jordan says political solution only path to st...,AMMAN (Reuters) - U.S. ally Jordan on Saturday...,https://www.reuters.com/article/us-mideast-cri...,reuters.com,None,None,1.0,100.0
2,Subaru says employees manipulated fuel economy...,Subaru says employees manipulated fuel economy...,https://www.cnbc.com/video/2018/04/27/subaru-s...,cnbc.com,None,None,1.0,95.0
3,Sgsco strengthens global position in package d...,"LOUISVILLE, Ky., Jan. 11, 2018 /PRNewswire/ --...",http://www.cnbc.com/2018/01/11/pr-newswire-sgs...,cnbc.com,None,2018-01-11 00:00:00,1.0,95.0
4,BRIEF-Smart Employee Benefits Q1 Rev $25.5 Mln...,April 30 (Reuters) - Smart Employee Benefits I...,https://www.reuters.com/article/brief-smart-em...,reuters.com,None,None,1.0,100.0
...,...,...,...,...,...,...,...,...
30619,Goldstrike Announces Proposed Spin-Off of Whit...,Highlights :\nGoldstrike's six 100% owned prop...,http://www.cnbc.com/2018/05/16/globe-newswire-...,cnbc.com,None,None,1.0,95.0
30620,BECLE S.A.B. de C.V. Announces Fourth Quarter ...,MEXICO CITY--(BUSINESS WIRE)-- BECLE S.A.B. de...,http://www.cnbc.com/2018/02/16/business-wire-b...,cnbc.com,None,2018-02-27 00:00:00,1.0,95.0
30621,Bannon steps down from Breitbart News,"Bannon steps down from Breitbart News Tuesday,...",https://www.reuters.com/video/2018/01/09/banno...,reuters.com,None,2018-01-09 00:00:00,1.0,100.0
30622,Factbox: Composition of Russia's new governmen...,MOSCOW (Reuters) - Russian Prime Minister Dmit...,https://www.reuters.com/article/us-russia-gove...,reuters.com,None,None,1.0,100.0


In [12]:
# Stream the ratings dataset
ds2 = load_dataset("sergioburdisso/news_media_reliability", split="train")

df2 = pd.DataFrame(ds2)

df2

,domain,reliability_label,newsguard_score
0,9news.com,1,NaN
1,nbc11news.com,1,NaN
2,12news.com,1,NaN
3,wibw.com,1,NaN
4,wifr.com,1,NaN
...,...,...,...
5327,chicago.suntimes.com,1,92.5
5328,instapundit.com,-1,42.0
5329,theantimedia.org,-1,20.0
5330,truepundit.com,-1,12.5
